# Day 1: Data Ingestion & Quality Exploration
**Project:** Bluestock Mutual Fund Analytics Platform  
**Deliverable:** 01_data_ingestion.ipynb  
**Tasks Covered:**
1. Ingest all 10 raw CSV datasets.
2. Inspect shapes, datatypes, and anomalies.
3. Fetch live NAV data from `mfapi.in` for HDFC Top 100 Direct and 5 key schemes.
4. Explore Fund Master structure and AMFI scheme code hierarchies.
5. Referential integrity validation across datasets.


In [ ]:
import os
import glob
import json
import requests
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 25)
pd.set_option('display.width', 1000)
print("Libraries loaded successfully.")


## 1. Load and Inspect 10 Provided CSV Datasets

In [ ]:
DATA_DIR = os.path.join("..", "data", "raw")
if not os.path.exists(DATA_DIR):
    DATA_DIR = os.path.join("data", "raw")

csv_files = [
    "01_fund_master.csv", "02_nav_history.csv", "03_aum_by_fund_house.csv",
    "04_monthly_sip_inflows.csv", "05_category_inflows.csv", "06_industry_folio_count.csv",
    "07_scheme_performance.csv", "08_investor_transactions.csv", "09_portfolio_holdings.csv",
    "10_benchmark_indices.csv"
]

dfs = {}
for fname in csv_files:
    fpath = os.path.join(DATA_DIR, fname)
    df = pd.read_csv(fpath)
    dfs[fname] = df
    null_count = int(df.isnull().sum().sum())
    dup_count = int(df.duplicated().sum())
    print(f"{fname:<30} | Shape: {str(df.shape):<15} | Nulls: {null_count:<5} | Duplicates: {dup_count}")


## 2. Fund Master Exploration
Exploring fund houses, categories, sub-categories, risk categories, and AMFI codes.


In [ ]:
master = dfs["01_fund_master.csv"]
print("--- Fund Houses (Top 10) ---")
print(master["fund_house"].value_counts())

print("
--- Sub-Categories ---")
print(master["sub_category"].value_counts())

print("
--- Risk Categories ---")
print(master["risk_category"].value_counts())


## 3. AMFI Code Referential Integrity Validation
Confirming every scheme code in `fund_master` exists in `nav_history`.


In [ ]:
nav = dfs["02_nav_history.csv"]
master_codes = set(master["amfi_code"].unique())
nav_codes = set(nav["amfi_code"].unique())

print("Unique schemes in fund_master :", len(master_codes))
print("Unique schemes in nav_history :", len(nav_codes))
print("Codes in master missing in nav:", master_codes - nav_codes)
print("Referential integrity check passed:", master_codes == nav_codes)


## 4. Live NAV Ingestion via mfapi.in API
Fetching live NAV for HDFC Top 100 Direct (125497) + 5 key schemes.


In [ ]:
schemes = {
    125497: "HDFC Top 100 Direct",
    119551: "SBI Bluechip",
    120503: "ICICI Bluechip",
    118632: "Nippon Large Cap",
    119092: "Axis Bluechip",
    120841: "Kotak Bluechip"
}

live_nav_records = []
for code, name in schemes.items():
    res = requests.get(f"https://api.mfapi.in/mf/{code}", timeout=10)
    payload = res.json()
    meta = payload.get("meta", {})
    history = payload.get("data", [])
    if history:
        latest = history[0]
        live_nav_records.append({
            "code": code,
            "target": name,
            "scheme_name": meta.get("scheme_name"),
            "fund_house": meta.get("fund_house"),
            "latest_date": latest["date"],
            "latest_nav": float(latest["nav"]),
            "historical_points": len(history)
        })

live_df = pd.DataFrame(live_nav_records)
print(live_df)
